# Semantic Similarity Analysis using Word Embeddings
## Retrieve the three closest words: 
GloVe(glove-wiki-gigaword-100, 400,000 vocabulary, 100-dimensional vectors) was loaded using the genesis library. The test of Pride and Prejudice was tokenised, resulting in 131,631 tokens and 6,982 unique types, of which 6,824 were present in the GloVe vocab. Nearest neighbours were captured using the cosine similarity, restricted to corpus words


In [1]:
!pip -q install --upgrade "scipy==1.16.3" "gensim==4.4.0"

In [2]:
# import libraries
import gensim.downloader as api 
import re
import numpy as np
import urllib.request
from collections import Counter
import sys
import subprocess
import importlib.util

# 2. Load pre-trained GloVe model
model = api.load("glove-wiki-gigaword-100")
print(f"  Vocabulary size: {len(model.key_to_index):,}\n")
URL = "https://www.gutenberg.org/files/1342/1342-0.txt"
with urllib.request.urlopen(URL) as response:
    raw_text = response.read().decode('utf-8')

# Tokenise
tokens = re.findall(r"[a-z]+", raw_text.lower())

# Unique vocabulary in corpus
corpus_vocab = set(tokens)

# Words in both corpus and model vocabulary
valid_words = corpus_vocab & set(model.key_to_index.keys())

print(f"  Corpus : Pride and Prejudice (Jane Austen)")
print(f"  Total tokens         : {len(tokens):,}")
print(f"  Unique word types    : {len(corpus_vocab):,}")
print(f"  Words also in model  : {len(valid_words):,}\n")


# Three nearest corpus neighbours per keyword
KEYWORDS = ['good', 'bad', 'happy', 'sad', 'angry']
TOP_N    = 3

def get_neighbours(keyword, model, valid_words, n=TOP_N):
   
    if keyword not in model.key_to_index:
        print(f"WARNING: '{keyword}' not in model vocabulary.")
        return []

    kw_vec  = model[keyword]
    kw_norm = np.linalg.norm(kw_vec)
    scored  = []

    for word in valid_words:
        if word == keyword:
            continue
        wv  = model[word]
        sim = np.dot(kw_vec, wv) / (kw_norm * np.linalg.norm(wv) + 1e-10)
        scored.append((word, float(sim)))

    scored.sort(key=lambda x: -x[1])
    return scored[:n]

# Collect results
results = {}
for kw in KEYWORDS:
    results[kw] = get_neighbours(kw, model, valid_words, TOP_N)
print("_" * 100)
print(" Three nearest neighbours - restricted to corpus vocab ")
print("_" * 100)

# Console table
print(f"\n  {'Keyword':<10}", end="")

for i in range(TOP_N):
    print(f"  {'Rank ' + str(i+1):<22}", end="")

print()

for kw in KEYWORDS:
    print(f"  {kw:<10}", end="")
    for word, sim in results[kw]:
        print(f"  {word:<14} ({sim:.4f})", end="")
    print()

freq = Counter(tokens)
print("\n\n Keyword frequencies in corpus")
for kw in KEYWORDS:
    print(f"    '{kw}' : {freq.get(kw, 0):4d} occurrences")


[==================================================] 100.0% 128.1/128.1MB downloaded
  Vocabulary size: 400,000

  Corpus : Pride and Prejudice (Jane Austen)
  Total tokens         : 128,577
  Unique word types    : 6,731
  Words also in model  : 6,577

____________________________________________________________________________________________________
 Three nearest neighbours - restricted to corpus vocab 
____________________________________________________________________________________________________

  Keyword     Rank 1                  Rank 2                  Rank 3                
  good        better         (0.8932)  sure           (0.8315)  really         (0.8298)
  bad         worse          (0.7930)  good           (0.7703)  things         (0.7654)
  happy       feel           (0.8133)  i              (0.7938)  really         (0.7904)
  sad         sorry          (0.7547)  awful          (0.7284)  horrible       (0.7049)
  angry       enraged        (0.7718)  frightened 